# Clean HubDailyContent

Cleans a raw `HubDailyContentData_yyyy-mm-dd.csv` export.

**Deviations / interpretations, mirroring the approach in `Clean_HubMonthlyUsers.ipynb`:**
- *"Remove completely blank rows"* is evaluated against the `LS_COLS` fields only, not the full raw column set — same reasoning as the Monthly notebook: raw pipeline-metadata columns (`DataSource`, `PipelineRunID`, `FileName`, ...) are dropped later and would otherwise mask genuinely blank rows.
- Only rows with a blank `Date` are dropped, per explicit instruction — matching the rule used in `Clean_HubDailyEvents.ipynb`/`Clean_HubDailyUsers.ipynb` (a `CompanyCode`-blank condition is not applied here).
- `ContentTitle` (raw, local-language title) is renamed to `ContentTitleLocalLanguage`, and `ContentTitleEN` (raw, English title) is renamed to `ContentTitle`, so the output matches `LS_COLS`, which lists both `ContentTitleLocalLanguage` and `ContentTitle` but has no `ContentTitleEN`. The notes' HTML-stripping step names the *raw* columns (`ContentTitle`, `ContentTitleEN`, `Stack`); this notebook applies that step to their renamed equivalents (`ContentTitleLocalLanguage`, `ContentTitle`, `Stack`) — the same columns, applied after the rename.
- HTML stripping removes tags (including ones with attributes, e.g. `<p style="...">`) via a generic `<...>` regex, then unescapes entities (e.g. `&nbsp;`, which occurs 396 times in the raw data) and collapses the resulting non-breaking spaces to plain spaces.
- The raw text is pre-processed before parsing to protect a literal backslash in the data (the real company name `TBWA\RAAD`) from being stripped by `escapechar` — see the note on the read-CSV cell below.
- The output filename follows the notes' worked example (`HubDailyContent_<month_tag>_cleaned.csv`), not the line just above it in the notes that says `HubMonthlyUsers_<month_tag>_cleaned.csv` — that line looks like a copy/paste leftover from the Monthly notes.
- The file is read and written with `encoding="utf-8"` explicitly, matching the Monthly notebook's reasoning (avoiding `cp1252` corruption of accented characters).


## Imports

In [1]:
import csv
import html
import io
import re
from pathlib import Path

import pandas as pd


## Schema constants

- `LS_COLS` — the final column set and order for the cleaned output.
- `LS_STRING_COLS` — the free-text columns that get whitespace-trimmed (per the notes, this also includes `Users`, `TotalEvents`, `UniqueEvents`, trimmed before being cast to integers below).
- `LS_INT_COLS` — the numeric columns that get cast to integer type.
- `HTML_COLS` — the columns that get HTML tags/entities stripped (the renamed equivalents of the raw `ContentTitle`, `ContentTitleEN`, `Stack` columns named in the notes).
- `RENAME_MAP` — raw → cleaned column renames (see the note at the top of this notebook).
- `FILENAME_RE` — extracts the year/month from the input filename, used both to build `month_tag` and to auto-detect the input file.
- `GROUP_COLS`/`SUM_COLS` — used to collapse duplicate rows (see "Collapse duplicate rows" below): every `LS_COLS` field except `LS_INT_COLS` is a group-by key, and `LS_INT_COLS` is what gets summed.


In [2]:
LS_COLS = [
    "Date", "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode", "Operation",
    "AppInterest", "CategoryName", "ContentTitleLocalLanguage", "ContentTitle", "ContentLanguage",
    "ContentType", "DeviceCategory", "UserType", "Users", "TotalEvents", "UniqueEvents",
    "Stack", "Route", "Topic",
]
LS_STRING_COLS = [
    "CompanyCode", "CompanyName", "Country", "HomeCountry", "HomeCountryCode", "Operation",
    "AppInterest", "CategoryName", "ContentTitleLocalLanguage", "ContentTitle", "ContentLanguage",
    "ContentType", "DeviceCategory", "UserType", "Users", "TotalEvents", "UniqueEvents",
    "Stack", "Route", "Topic",
]
LS_INT_COLS = ["Users", "TotalEvents", "UniqueEvents"]
HTML_COLS = ["ContentTitleLocalLanguage", "ContentTitle", "Stack"]

RENAME_MAP = {"ContentTitle": "ContentTitleLocalLanguage", "ContentTitleEN": "ContentTitle"}

FILENAME_RE = re.compile(r"^HubDailyContentData_(\d{4})-(\d{2})-\d{2}\.csv$")

# Duplicate rows are collapsed by grouping on every LS_COLS field except the summed
# measures (Users/TotalEvents/UniqueEvents) and summing those.
GROUP_COLS = [c for c in LS_COLS if c not in LS_INT_COLS]
SUM_COLS = LS_INT_COLS


## Locate the input file

`find_default_input` looks for a single `HubDailyContentData_yyyy-mm-dd.csv` file in a given directory and returns it automatically. If none or several are found, it raises rather than silently guessing which one to use.


In [3]:
def find_default_input(directory: Path) -> Path:
    matches = sorted(p for p in directory.glob("HubDailyContentData_*.csv") if FILENAME_RE.match(p.name))
    if not matches:
        raise FileNotFoundError(f"No HubDailyContentData_yyyy-mm-dd.csv file found in {directory}")
    if len(matches) > 1:
        raise ValueError(
            f"Multiple candidate input files found in {directory}: "
            f"{[m.name for m in matches]}. Pass one explicitly."
        )
    return matches[0]


## Derive `month_tag` from the filename

The output name has the format of `HubDailyContent_<month_tag>_cleaned.csv`, where `month_tag` is `yyyymm` for the month *before* the input filename's `yyyy-mm-dd` date suffix (the export date's month minus one), per the notes' worked example.


In [4]:
def month_tag_from_filename(path: Path) -> str:
    match = FILENAME_RE.match(path.name)
    if not match:
        raise ValueError(f"Filename '{path.name}' does not match expected pattern HubDailyContentData_yyyy-mm-dd.csv")
    year, month = (int(g) for g in match.groups())
    # month_tag refers to the prior month's data, not the export date's month.
    year, month = (year - 1, 12) if month == 1 else (year, month - 1)
    return f"{year}{month:02d}"


## Cleaning logic

The core transformation, in the order implemented (the notes list these unordered):

1. Rename `ContentTitle` → `ContentTitleLocalLanguage` and `ContentTitleEN` → `ContentTitle` (see the note at the top of this notebook).
2. Drop rows that are blank across every `LS_COLS` field.
3. Drop rows where `Date` is blank.
4. Reformat `Date` to `%Y-%m-%d %H:%M:%S.%f` truncated to 3 decimals (milliseconds), matching the Monthly notebook's convention.
5. Reorder/drop columns to match `LS_COLS`.
6. Trim surrounding whitespace on the `LS_STRING_COLS` fields.
7. Cast `Users`, `TotalEvents`, `UniqueEvents` to integer type.
8. Strip HTML tags/attributes and unescape HTML entities on `ContentTitleLocalLanguage`, `ContentTitle`, `Stack`.


In [5]:
HTML_TAG_RE = re.compile(r"<[^>]+>")


def strip_html(value: str) -> str:
    text = HTML_TAG_RE.sub("", value)
    text = html.unescape(text)
    return text.replace("\xa0", " ").strip()


def clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns=RENAME_MAP)

    # Same reasoning as Clean_HubMonthlyUsers: judge "completely blank" against the
    # LS_COLS fields only, since raw pipeline-metadata columns (DataSource, PipelineRunID,
    # FileName, ...) are dropped later and would otherwise mask genuinely blank rows.
    present_ls_cols = [c for c in LS_COLS if c in df.columns]
    is_blank = df[present_ls_cols].apply(lambda col: col.str.strip() == "").all(axis=1)
    df = df.loc[~is_blank].copy()

    # Only Date being blank drops a row, matching Clean_HubDailyEvents/Clean_HubDailyUsers.
    missing_key = df["Date"].str.strip() == ""
    df = df.loc[~missing_key].copy()

    # %f always zero-pads to 6-digit microseconds; slicing off the last 3 leaves milliseconds.
    df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d").dt.strftime("%Y-%m-%d %H:%M:%S.%f").str[:-3]

    df = df[[c for c in LS_COLS if c in df.columns]]

    for col in LS_STRING_COLS:
        df[col] = df[col].str.strip().str.replace(r"\s+", " ", regex=True)

    for col in LS_INT_COLS:
        df[col] = df[col].astype(int)

    for col in HTML_COLS:
        df[col] = df[col].apply(strip_html)

    return df


## Collapse duplicate rows

Duplicates are not allowed in the final cleaned dataset. Rows that share every `GROUP_COLS` value (i.e. every `LS_COLS` field except `Users`/`TotalEvents`/`UniqueEvents`) are collapsed into one row, summing those three measures. `Users`/`TotalEvents`/`UniqueEvents` are re-cast to integer defensively (they're already int by this point in `clean()`) so the sum is numeric, not string concatenation.


In [6]:
def collapse_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in SUM_COLS:
        df[col] = df[col].astype(int)
    df = df.groupby(GROUP_COLS, as_index=False)[SUM_COLS].sum()
    return df[LS_COLS]


## Configure the input file

Leave `INPUT_FILE` as `None` to auto-detect the single raw file in this notebook's `input/` folder, or set it to an explicit path to override (equivalent to the script's optional CLI argument).


In [7]:
NOTEBOOK_DIR = Path.cwd()
INPUT_FILE = None  # e.g. "input/HubDailyContentData_2026-08-02.csv"

input_path = Path(INPUT_FILE).resolve() if INPUT_FILE else find_default_input(NOTEBOOK_DIR / "input")
month_tag = month_tag_from_filename(input_path)
input_path, month_tag


(WindowsPath('C:/Users/HenriBranken/Documents/Automation/Gabriella__HubMercury/HUB/input/HubDailyContentData_2026-08-02.csv'),
 '202607')

## Read the raw CSV

Read everything as strings (`dtype=str`, `keep_default_na=False`) so blank fields and numeric-looking codes pass through unchanged instead of being coerced or turned into `NaN`. `engine="python"` with `escapechar="\\"` (per the notes) handles the backslash-escaped quotes inside the HTML-bearing text fields. `encoding="utf-8"` matches the source file and avoids corrupting accented text.

Before parsing, the raw text is pre-processed to double any backslash that isn't immediately followed by `"`. Without this, `escapechar` strips *every* backslash it precedes, not just ones before a quote — and the raw data contains a real company name, `TBWA\RAAD`, whose backslash isn't a CSV escape artifact at all. The doubling makes `escapechar` only ever consume genuine `\"` sequences, so `TBWA\RAAD` survives intact while the legitimate HTML-attribute escaping (e.g. `<p style=\"...\">`) still parses correctly.


In [8]:
with open(input_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

# Protect literal backslashes that aren't a genuine CSV \" escape (e.g. the real
# company name "TBWA\RAAD") by doubling them, so escapechar below only ever consumes
# actual \" sequences and every other backslash survives as a literal single backslash.
protected_text = re.sub(r'\\(?!")', r"\\\\", raw_text)

df_raw = pd.read_csv(
    io.StringIO(protected_text), sep=";", engine="python", escapechar="\\",
    dtype=str, keep_default_na=False, encoding="utf-8",
)
df_raw.shape


(10102, 30)

## Apply the cleaning steps

In [9]:
df_cleaned = clean(df_raw)
df_cleaned.head()


,Date,CompanyCode,CompanyName,Country,HomeCountry,HomeCountryCode,Operation,AppInterest,CategoryName,ContentTitleLocalLanguage,...,ContentLanguage,ContentType,DeviceCategory,UserType,Users,TotalEvents,UniqueEvents,Stack,Route,Topic
0,2026-07-06 00:00:00.000,ROCHE,ROCHE,Algeria,Algeria,DZ,Lyra Health International Ltd,Relationships,,Gérer ses émotions pendant une rupture,...,fr,Article,Desktop,Returning User,1,1,1,Impactful life changes,Explore,Divorce and break-up
1,2026-07-01 00:00:00.000,TETRAPAK,Tetrapak,Tunisia,Algeria,DZ,Lyra Health International Ltd,"Lifestyle,Relationships",,Comment résoudre les conflits avec les collègues,...,fr,Article,Mobile,Returning User,1,1,1,Interpersonal skills,Explore,Interpersonal skills
2,2026-07-07 00:00:00.000,MOODYS,Moody's Shared Services,Germany,Lithuania,LT,Lyra Health International Ltd,MentalHealth,,Returning to work after a major life change,...,en,Article,Desktop,Returning User,1,1,1,Thriving at work,Explore,Adapting to change
3,2026-07-31 00:00:00.000,BEIERSDORF,BEIERSDORF,Finland,Finland,FI,Lyra Health International Ltd,Lifestyle,Mental Health,How to motivate yourself when you're strugglin...,...,en,Article,Desktop,Returning User,1,1,1,Achieving your goals,Explore,Goal setting
4,2026-07-31 00:00:00.000,BEIERSDORF,BEIERSDORF,Finland,Finland,FI,Lyra Health International Ltd,Relationships,,6 ways to build a stronger team,...,en,Article,Desktop,Returning User,1,1,1,,Explore,"Leadership,Team dynamics,Teambuilding"


## Collapse duplicate rows before saving

`rows_before_dedup` is kept so the report below can still report "blank/missing-key rows dropped" against the pre-dedup count, separately from rows collapsed for being duplicates.


In [10]:
rows_before_dedup = len(df_cleaned)
df_cleaned = collapse_duplicates(df_cleaned)
duplicates_collapsed = rows_before_dedup - len(df_cleaned)

print(f"Collapsed {duplicates_collapsed} duplicate rows -> {len(df_cleaned)} rows remaining")


Collapsed 113 duplicate rows -> 9985 rows remaining


## Save the cleaned dataset

Written as `;`-delimited UTF-8 with minimal quoting, matching the input file's own semicolon delimiter as required by the notes. Saved to this notebook's `output/` folder.


In [11]:
output_dir = NOTEBOOK_DIR / "output"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / f"HubDailyContent_{month_tag}_cleaned.csv"
df_cleaned.to_csv(output_path, sep=";", index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL)

print(f"Cleaned {len(df_cleaned)} rows -> {output_path}")


Cleaned 9985 rows -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\output\HubDailyContent_202607_cleaned.csv


## Write summary report

Writes a plain-text report answering: which input file was read, the raw and cleaned row counts, how many duplicate rows exist in each of the raw and cleaned dataframes, the derived `month_tag`, how many rows were dropped for being blank / missing their key fields, and the output CSV's name, followed (after three blank lines) by `df_cleaned.describe()`. Saved to this notebook's `reports/` folder as `HubDailyContent_<month_tag>_report.txt`.

Duplicate counts use pandas' default `duplicated()` (`keep="first"`), i.e. the number of rows that are repeats of an earlier row — how many rows would go away if the dataframe were deduplicated.


In [12]:
report_lines = [
    f"Input file: {input_path.name}",
    f"Raw row count: {len(df_raw)}",
    f"Raw duplicate rows: {int(df_raw.duplicated().sum())}",
    "=======================================================================",
    f"Month tag: {month_tag}",
    f"Blank/missing-key rows dropped: {len(df_raw) - rows_before_dedup}",
    f"Duplicate rows collapsed: {duplicates_collapsed}",
    "=======================================================================",
    f"Cleaned row count: {len(df_cleaned)}",
    f"Cleaned duplicate rows: {int(df_cleaned.duplicated().sum())}",
    f"Output file: {output_path.name}",
]
report_text = "\n".join(report_lines) + "\n"
report_text += "\n\n\n" + df_cleaned.describe().to_string() + "\n"

reports_dir = NOTEBOOK_DIR / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)
report_path = reports_dir / f"HubDailyContent_{month_tag}_report.txt"
report_path.write_text(report_text, encoding="utf-8")

print(report_text)
print(f"Report written -> {report_path}")


Input file: HubDailyContentData_2026-08-02.csv
Raw row count: 10102
Raw duplicate rows: 73
Month tag: 202607
Blank/missing-key rows dropped: 4
Duplicate rows collapsed: 113
Cleaned row count: 9985
Cleaned duplicate rows: 0
Output file: HubDailyContent_202607_cleaned.csv



             Users  TotalEvents  UniqueEvents
count  9985.000000  9985.000000   9985.000000
mean      1.020831     1.104857      1.040661
std       0.165561     0.445914      0.257011
min       1.000000     1.000000      1.000000
25%       1.000000     1.000000      1.000000
50%       1.000000     1.000000      1.000000
75%       1.000000     1.000000      1.000000
max       6.000000    16.000000     10.000000

Report written -> C:\Users\HenriBranken\Documents\Automation\Gabriella__HubMercury\HUB\reports\HubDailyContent_202607_report.txt
